# Stage 4 — Domain-Adapt Fine-Tune: OCR/Tag-Reading Task (Qwen3-VL)

Self-contained. This is the **deferred task-neutral domain-adaptation pass** (separate
checkpoint from the combined detection fine-tune) — branches fresh from the original
Qwen3-VL base, per the 2026-07-10 resequencing decision.

**Training signal:** real Tesseract OCR output over Gupta's 72 train tiles, used as
**machine-generated pseudo-labels** — not human-verified ground truth. Gupta itself has
no real tag-transcription data (checked the full archive directly: no CSV/JSON text
labels anywhere, just symbol crops + detection labels). Caveat this clearly in any
write-up — OCR errors become training noise, not truth.

**Built to run unattended overnight:** checkpoints to Drive every epoch, resumes from
the latest checkpoint automatically if the runtime disconnects and you reconnect and
re-run this notebook.

## 1. Mount Drive (needed for overnight checkpoint persistence)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Install dependencies (transformers stack + Tesseract OCR)

In [ ]:
!apt-get -qq install -y tesseract-ocr
!pip install -q transformers accelerate peft pytesseract qwen-vl-utils

## 3. Config + imports

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/pid_project/data"
GUPTA_DIR = "gupta_pid"
CHECKPOINT_DIR = "/content/drive/MyDrive/pid_project/checkpoints/qwen3vl_domain_adapt_ocr"

import json, time, random
from pathlib import Path
import torch
import pytesseract
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

ROOT = Path(DRIVE_ROOT)
gupta_p = ROOT / GUPTA_DIR
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
print("Gupta:", gupta_p.exists())
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU — check Runtime type before continuing"

## 4. Tiling (checklist 2.3) + OCR pseudo-label generation

Runs Tesseract over each Gupta train tile, keeps text strings that look like real tags
(alphanumeric, length >= 2 — filters out single-character OCR noise from line art).

In [ ]:
TILE_SIZE, OVERLAP = 1024, 205

def compute_tile_grid(img_w, img_h, tile_size=TILE_SIZE, overlap=OVERLAP):
    stride = tile_size - overlap
    tiles = []
    y0 = 0
    while y0 < img_h:
        y1 = min(y0 + tile_size, img_h)
        x0 = 0
        while x0 < img_w:
            x1 = min(x0 + tile_size, img_w)
            tiles.append((x0, y0, x1, y1))
            x0 += stride
        y0 += stride
    return tiles

def ocr_tags(tile_img, min_len=2):
    """Real Tesseract OCR — output is a pseudo-label, not verified ground truth."""
    raw_text = pytesseract.image_to_string(tile_img)
    tokens = [t.strip() for t in raw_text.split() if len(t.strip()) >= min_len]
    # keep tokens that look like real tags: has a digit or a hyphen (P&ID tags are things
    # like "V-001", "CS-08", "8\"-IC-3076") - filters generic OCR garbage from line art
    tags = [t for t in tokens if any(c.isdigit() for c in t) or "-" in t]
    return sorted(set(tags))

def build_ocr_examples(max_sheets=72, max_tiles_per_sheet=None):
    raw = gupta_p / "PID_Dataset" / "0__raw_data"
    sheet_paths = sorted((raw / "sheets" / "train").glob("*.*"))[:max_sheets]
    examples = []
    for sheet_path in sheet_paths:
        img = Image.open(sheet_path).convert("RGB")
        W, H = img.size
        tiles = compute_tile_grid(W, H)
        if max_tiles_per_sheet:
            tiles = tiles[:max_tiles_per_sheet]
        for (x0, y0, x1, y1) in tiles:
            crop = img.crop((x0, y0, x1, y1))
            tags = ocr_tags(crop)
            if not tags:
                continue  # skip tiles with no readable text — not informative for this task
            target_text = "Tags visible in this tile: " + ", ".join(tags)
            examples.append({"image_path": str(sheet_path), "crop": [x0, y0, x1, y1], "target_text": target_text})
    return examples

print("Building OCR pseudo-labeled training set (this scans all 72 train sheets — may take a few minutes)...")
t0 = time.time()
ocr_examples = build_ocr_examples()
print(f"Built {len(ocr_examples)} training examples in {time.time()-t0:.1f}s")
print("Sample:", ocr_examples[0] if ocr_examples else "(none found)")

## 5. Load Qwen3-VL, apply LoRA

In [ ]:
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
OCR_PROMPT = "List every text tag or label you can read in this P&ID tile."

processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(QWEN_MODEL_ID, dtype=torch.bfloat16, device_map="cuda")
print("Base loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Resume from latest checkpoint, if any (for overnight-disconnect recovery)

In [ ]:
start_epoch = 0
ckpt_path = Path(CHECKPOINT_DIR)
existing = sorted(ckpt_path.glob("epoch_*"), key=lambda p: int(p.name.split("_")[1]))
if existing:
    latest = existing[-1]
    start_epoch = int(latest.name.split("_")[1]) + 1
    model.load_adapter(str(latest), adapter_name="default", is_trainable=True)
    print(f"Resumed from {latest}, continuing at epoch {start_epoch}")
else:
    print("No existing checkpoint — starting from epoch 0")

## 7. Training loop — checkpoints every epoch, safe to leave running overnight

Wraps each step in try/except so one bad example (e.g. a corrupt OCR result) can't kill
an unattended multi-hour run.

In [ ]:
N_EPOCHS = 3
LR = 1e-4

def build_labeled_inputs(image, prompt, target_text):
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    prompt_inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
    )
    prompt_len = prompt_inputs["input_ids"].shape[1]
    full_messages = messages + [{"role": "assistant", "content": [{"type": "text", "text": target_text}]}]
    full_inputs = processor.apply_chat_template(
        full_messages, tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors="pt",
    )
    labels = full_inputs["input_ids"].clone()
    labels[:, :prompt_len] = -100
    full_inputs["labels"] = labels
    return {k: v.to(model.device) for k, v in full_inputs.items()}

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)

for epoch in range(start_epoch, N_EPOCHS):
    random.seed(epoch)
    order = list(range(len(ocr_examples)))
    random.shuffle(order)
    epoch_losses = []
    t0 = time.time()
    for step, idx in enumerate(order):
        ex = ocr_examples[idx]
        try:
            img = Image.open(ex["image_path"]).convert("RGB").crop(ex["crop"])
            inputs = build_labeled_inputs(img, OCR_PROMPT, ex["target_text"])
            out = model(**inputs)
            loss = out.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            epoch_losses.append(loss.item())
        except Exception as e:
            print(f"  [skip] step {step} failed: {e}")
            continue
        if step % 50 == 0:
            avg = sum(epoch_losses[-50:]) / len(epoch_losses[-50:]) if epoch_losses else float("nan")
            print(f"epoch {epoch} step {step}/{len(order)} avg_loss={avg:.4f} elapsed={time.time()-t0:.0f}s")

    epoch_ckpt = ckpt_path / f"epoch_{epoch}"
    model.save_pretrained(str(epoch_ckpt))
    mean_loss = sum(epoch_losses) / len(epoch_losses) if epoch_losses else float("nan")
    print(f"=== epoch {epoch} done: mean_loss={mean_loss:.4f}, saved to {epoch_ckpt} ===\n")

print("Training complete.")